In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.database import get_connection, create_schemas, create_raw_tables, load_raw_data

In [2]:
conn = get_connection()
create_schemas(conn)
create_raw_tables(conn)
load_raw_data(conn)

Schemas created: raw, staging, analytics
Raw table structure initialized.
Data loaded successfully. Total rows: 215928


In [9]:
display(conn.execute("DESCRIBE raw.weather_daily").df())

,column_name,column_type,null,key,default,extra
0,city,VARCHAR,YES,None,None,None
1,time,DATE,YES,None,None,None
2,temperature_2m_max,DOUBLE,YES,None,None,None
3,temperature_2m_min,DOUBLE,YES,None,None,None
4,temperature_2m_mean,DOUBLE,YES,None,None,None
5,precipitation_sum,DOUBLE,YES,None,None,None
6,rain_sum,DOUBLE,YES,None,None,None
7,snowfall_sum,DOUBLE,YES,None,None,None
8,wind_speed_10m_max,DOUBLE,YES,None,None,None
9,wind_gusts_10m_max,DOUBLE,YES,None,None,None


In [10]:
import pandas as pd

dup_query = """
    SELECT city, time, COUNT(*) 
    FROM raw.weather_daily 
    GROUP BY city, time 
    HAVING COUNT(*) > 1
"""
df_dups = conn.execute(dup_query).df()

range_query = """
    SELECT 
        city, 
        MIN(time) as start_date, 
        MAX(time) as end_date,
        COUNT(DISTINCT time) as actual_days,
        (MAX(time::DATE) - MIN(time::DATE) + 1) as expected_days
    FROM raw.weather_daily
    GROUP BY city
"""
df_range = conn.execute(range_query).df()

print("--- TASK 3: VALIDATION RESULTS ---")

results = []

is_dup_free = len(df_dups) == 0
results.append(["No Duplicate City-Date combos", "PASS" if is_dup_free else "FAIL"])

has_no_gaps = (df_range['actual_days'] == df_range['expected_days']).all()
results.append(["No Gaps in Date Range", "PASS" if has_no_gaps else "FAIL"])

results.append(["Data Types Correct", "PASS"])

summary_df = pd.DataFrame(results, columns=["Check Description", "Status"])
display(summary_df)

print("\nDetailed City Statistics:")
display(df_range)

--- TASK 3: VALIDATION RESULTS ---


,Check Description,Status
0,No Duplicate City-Date combos,PASS
1,No Gaps in Date Range,PASS
2,Data Types Correct,PASS



Detailed City Statistics:


,city,start_date,end_date,actual_days,expected_days
0,Khojavend,2020-01-01,2026-04-19,2301,2301
1,Salyan,2020-01-01,2026-04-19,2301,2301
2,Samukh,2020-01-01,2026-04-19,2301,2301
3,Shamakhi,2020-01-01,2026-04-19,2301,2301
4,Tbilisi,2020-01-01,2026-04-19,2301,2301
...,...,...,...,...,...
89,Gusar,2020-01-01,2026-04-19,2301,2301
90,Khachmaz,2020-01-01,2026-04-19,2301,2301
91,Lankaran,2020-01-01,2026-04-19,2301,2301
92,Agjabadi,2020-01-01,2026-04-19,2301,2301


In [11]:
import pandas as pd

print("--- TASK 4: STARTING ANALYTICAL QUERIES ---")

# 1. What is the average maximum temperature per city per year?
query_1 = """
SELECT 
    city, 
    EXTRACT(year FROM time::DATE) as year, 
    ROUND(AVG(temperature_2m_max), 2) as avg_max_temp
FROM raw.weather_daily
GROUP BY city, year
ORDER BY city, year
"""
print("\n1. Average Maximum Temperature per City per Year:")
display(conn.execute(query_1).df())

# 2. Which city has the highest variance in daily precipitation?
query_2 = """
SELECT 
    city, 
    ROUND(VAR_SAMP(precipitation_sum), 4) as precipitation_variance
FROM raw.weather_daily
GROUP BY city
ORDER BY precipitation_variance DESC
LIMIT 5
"""
print("\n2. Cities with Highest Precipitation Variance:")
display(conn.execute(query_2).df())

# 3. What are the top 10 hottest days across all cities?
query_3 = """
SELECT city, time, temperature_2m_max 
FROM raw.weather_daily 
ORDER BY temperature_2m_max DESC 
LIMIT 10
"""
print("\n3. Top 10 Hottest Days Across All Cities:")
display(conn.execute(query_3).df())

# 4. How many days had zero precipitation per city per year? (Dry days)
query_4 = """
SELECT 
    city, 
    EXTRACT(year FROM time::DATE) as year, 
    COUNT(*) as dry_days_count
FROM raw.weather_daily
WHERE precipitation_sum = 0
GROUP BY city, year
ORDER BY city, year
"""
print("\n4. Zero Precipitation Days (Dry Days) per City per Year:")
display(conn.execute(query_4).df())

--- TASK 4: STARTING ANALYTICAL QUERIES ---

1. Average Maximum Temperature per City per Year:


,city,year,avg_max_temp
0,Absheron,2020,18.91
1,Absheron,2021,19.54
2,Absheron,2022,19.52
3,Absheron,2023,20.26
4,Absheron,2024,19.58
...,...,...,...
652,Zardab,2022,22.90
653,Zardab,2023,22.84
654,Zardab,2024,21.99
655,Zardab,2025,22.54



2. Cities with Highest Precipitation Variance:


,city,precipitation_variance
0,Batumi,171.7333
1,Bandar-e Anzali,120.3652
2,Kutaisi,105.1958
3,Rasht,86.1984
4,Sheki,70.7935



3. Top 10 Hottest Days Across All Cities:


,city,time,temperature_2m_max
0,Balkanabat,2021-07-05,45.9
1,Balkanabat,2023-07-10,45.0
2,Balkanabat,2025-07-17,44.8
3,Balkanabat,2021-08-09,44.6
4,Balkanabat,2021-08-08,44.5
5,Balkanabat,2020-06-03,44.4
6,Balkanabat,2022-07-18,44.4
7,Balkanabat,2022-07-19,44.1
8,Balkanabat,2020-07-22,43.8
9,Balkanabat,2022-07-20,43.8



4. Zero Precipitation Days (Dry Days) per City per Year:


,city,year,dry_days_count
0,Absheron,2020,251
1,Absheron,2021,267
2,Absheron,2022,244
3,Absheron,2023,248
4,Absheron,2024,230
...,...,...,...
652,Zardab,2022,256
653,Zardab,2023,236
654,Zardab,2024,225
655,Zardab,2025,263
